# BanglaGPT-Pro — full fine-tune of Gemma-3-1B for Bangla, end to end

**Kaggle, GPU T4 ×2.** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.
`HF_TOKEN` as a Kaggle Secret (Add-ons → Secrets).
Then **Save Version → Save & Run All (Commit)** — not plain Run All, see below.

This notebook trains the model, evaluates it, builds the GGUF files and writes the model
card. It is designed to be run **more than once**: 80,000 examples do not fit in one
9-hour Kaggle session, so it checkpoints to a private Hub repo from inside the training
loop and picks up where it left off. When training reaches the target it moves straight
on to evaluation and publishing in the same session.

### Run it with Save & Run All, not Run All

Interactive sessions are killed after 20–40 minutes without a browser connection, so a
closed laptop ends the run. Save & Run All executes in the background: the tab can be
closed, the machine can sleep, and the session gets its full 9 hours.

Progress survives either way — every checkpoint is pushed to the Hub as it is written, so
an abort costs at most `CKPT_EVERY` steps and **no later cell has to run** for that to
hold. But Save & Run All avoids the situation entirely.

### What happens, in order

| Stage | |
|---|---|
| Pre-flight | all five benchmark adapters validated before a single GPU-hour is spent |
| Census | how many of all 342,391 rows survive the 512-token filter, and how many of the slice |
| Train | **full fine-tune**, every parameter, 80,000 examples, resumable across sessions |
| Evaluate | base and fine-tune on five benchmarks at **1,000 items**, plus perplexity |
| Publish | merged weights, GGUF for llama.cpp, and a card built from the numbers |

### How the evaluation works

Bangla generation is hard to score automatically, so the headline numbers come from
**multiple-choice benchmarks scored by length-normalised log-likelihood**: for each
option, the mean log probability of its tokens given the prompt, highest wins. No
generation, no judge, no string matching.

At 1,000 items the standard error of a difference between two models is ±0.022, so
anything smaller than about 0.045 is noise — and the card labels it as such rather than
reporting it as a result.

### What is measured here, and what is only quoted

**Measured: two models.** `gemma-3-1b-it` and this fine-tune, same items, same scorer. The
base is not a competitor; it is the before-picture, and without it there is no Δ column.

**Quoted: everyone else.** TituLLM's published figures are on exactly these five datasets
— TituLLM built them — but were produced with `lm-evaluation-harness`, which sends the raw
question with no chat template and normalises by character length, where this notebook
uses the chat template with a Bangla system prompt and normalises by token count. Same
data, different measurement; those columns are marked † and are for scale, not ranking.
TigerLLM's figures are on six datasets, none of them these, so its table goes in an
appendix with the problems stated before the numbers and no row for this model.

## 1. Environment

`/kaggle/working` is capped at 20 GB. A full-finetune checkpoint is ~4.6 GB and Trainer
keeps two on disk while rotating, so the Hub cache and the resume copy must live
elsewhere or the optimizer save runs out of room half-way through `optimizer.pt` — a
`torch.save` "unexpected pos" error that takes the whole run down with it.

In [ ]:
import subprocess, os, time, shutil

SESSION_T0 = time.time()
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                               "--format=csv,noheader"]).decode().strip())
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # Unsloth is single-GPU; T4 #2 stays idle
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def _pick_scratch():
    best, best_free = None, 0
    for cand in ("/kaggle/temp", "/tmp", "/kaggle/working"):
        try:
            os.makedirs(cand, exist_ok=True)
            f = shutil.disk_usage(cand).free
            print(f"  {cand:18} {f/1e9:6.1f} GB free")
            if f > best_free:
                best, best_free = cand, f
        except Exception:
            pass
    return best

print("scratch candidates:")
SCRATCH = _pick_scratch()
os.environ["HF_HOME"] = f"{SCRATCH}/hf"
os.environ["HF_HUB_CACHE"] = f"{SCRATCH}/hf/hub"
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
print(f"Hub cache and resume copy -> {SCRATCH}")
print("free in /kaggle/working:",
      round(shutil.disk_usage("/kaggle/working").free / 1e9, 1), "GB")

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps --upgrade git+https://github.com/unslothai/unsloth.git
!pip install -q matplotlib
print("installed")

## 2. Configuration

**Full fine-tuning on a 16 GB T4.** Gemma-3-1B is about 2 GB in fp16 and gradients add
2 GB. Plain AdamW would need ~8 GB more for its two fp32 moment buffers, which does not
fit; `adamw_8bit` holds them in 8 bits for roughly 2 GB, and gradient checkpointing
absorbs the activations. At 3B this would not fit without quantising the base, at which
point it is QLoRA, not a full fine-tune.

**Changing the training size.** `TARGET_EXAMPLES` is the only number to touch;
`TOTAL_STEPS` follows from it. `SELECT_N` must stay above it with room for the two
filters (~70% survive), and section 8b asserts this rather than quietly training on less.
Change either **before the first session** — a checkpoint written under a different target
is not resumable, and section 4 refuses it rather than corrupting the schedule.

In [ ]:
import gc, re, json, random, inspect, math, glob
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from unsloth import FastModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer
from transformers import TrainerCallback

MODEL_NAME  = "BanglaGPT-Pro"
HF_USER     = "emonHasan01"
BASE_MODEL  = "unsloth/gemma-3-1b-it"          # ungated mirror of the upstream model
UPSTREAM    = "google/gemma-3-1b-it"
SFT_DATASET = "md-nishat-008/Bangla-Instruct"

# --- training target -------------------------------------------------------------------
MAX_SEQ         = 512
BATCH           = 2
ACCUM           = 8                            # effective 16
TARGET_EXAMPLES = 80_000                       # what the model must actually SEE
TOTAL_STEPS     = math.ceil(TARGET_EXAMPLES / (BATCH * ACCUM))   # 5,000
SELECT_N        = 115_000                      # raw rows pulled before filtering
LR              = 1e-5                         # full fine-tune: an order below LoRA's 2e-4
SEED            = 3407

# --- session budget --------------------------------------------------------------------
SESSION_LIMIT_MIN = 8.5 * 60                   # Kaggle hard-kills at 9 h
RESERVE_MIN       = 200                        # eval of two models + GGUF + upload
CKPT_EVERY        = 500                        # steps between checkpoints, each pushed

# --- evaluation ------------------------------------------------------------------------
BENCH_N    = 1000                              # per benchmark
TOK_BUDGET = 2500                              # tokens per batched scoring forward
PPL_N      = 200
PPL_FROM, PPL_TO = 200_000, 200_800            # far past the training slice

FULL_CENSUS = True                             # tokenise all 342k rows to report coverage

WORK   = "/kaggle/working"
SAVE   = f"{WORK}/{MODEL_NAME}"
MERGED = f"{WORK}/{MODEL_NAME}-merged"
GGUF   = f"{WORK}/{MODEL_NAME}-GGUF"
ASSETS = f"{SAVE}/assets"
OUTDIR = f"{WORK}/outputs"
CKPT_LOCAL = f"{SCRATCH}/resume"               # NOT under WORK
for d in (SAVE, ASSETS):
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
GPU, BF16 = torch.cuda.get_device_name(0), is_bfloat16_supported()
print(f"{GPU} | bfloat16 {BF16} -> training in {'bf16' if BF16 else 'fp16'}")
print(f"target {TARGET_EXAMPLES:,} examples = {TOTAL_STEPS:,} steps at "
      f"{BATCH}x{ACCUM}={BATCH*ACCUM} effective")
print(f"eval {BENCH_N:,} items/benchmark, SE of a difference "
      f"±{math.sqrt(2)*0.5/math.sqrt(BENCH_N):.3f}")

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 130, "savefig.bbox": "tight",
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.facecolor": "white"})
C_BASE, C_TUNED, C_OTHER = "#94a3b8", "#059669", "#c7d2fe"

SYSTEM_BN = "আপনি একজন সহায়ক সহকারী। সবসময় স্পষ্ট ও সঠিক বাংলায় উত্তর দিন।"
PROBES = [
    "বাংলাদেশের মুক্তিযুদ্ধ সম্পর্কে তিনটি বাক্যে লিখুন।",
    "রবীন্দ্রনাথ ঠাকুর কে ছিলেন?",
    "একটি ছোট গল্প লিখুন যেখানে একজন কৃষক বৃষ্টির জন্য অপেক্ষা করছেন।",
    "পানি ফুটলে কী ঘটে, সহজ ভাষায় ব্যাখ্যা করুন।",
]

def to_chat(user, assistant=None, system=SYSTEM_BN):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user})
    if assistant is not None:
        msgs.append({"role": "assistant", "content": assistant})
    return msgs

def vram(): return torch.cuda.max_memory_allocated() / 1e9
def clear(): gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def elapsed_min(): return (time.time() - SESSION_T0) / 60

## 3. Banner

The card opens with `assets/banner.png` if one is present and silently drops the `<img>`
line if not. Upload the banner as a Kaggle Dataset and it will be found here.

In [ ]:
BANNER = None
for pat in (f"/kaggle/input/**/{MODEL_NAME}.png", "/kaggle/input/**/banner.png",
            f"{WORK}/{MODEL_NAME}.png", f"{WORK}/banner.png"):
    hits = glob.glob(pat, recursive=True)
    if hits:
        BANNER = hits[0]; break
if BANNER:
    shutil.copy(BANNER, f"{ASSETS}/banner.png")
    print(f"banner: {BANNER} -> {ASSETS}/banner.png")
else:
    print("no banner found; the card's <img> line will be removed automatically")
    print("  (upload BanglaGPT-Pro.png as a Kaggle Dataset to include it)")

## 4. Hugging Face, checkpoints and caches

One private repo holds three things: the resumable training checkpoint, the cached
evaluation of each model, and a marker saying training finished. All of it exists only so
a run can span sessions; delete it once the model is published.

A checkpoint written under a different `TARGET_EXAMPLES` cannot be resumed — the cosine
schedule was built for a different horizon, so the learning rate would jump. The cell
refuses rather than resuming into a corrupted schedule.

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import (login, create_repo, upload_folder, upload_file,
                             snapshot_download, hf_hub_download, whoami)
login(token=HF_TOKEN)
WHO = whoami()["name"]
if WHO != HF_USER:
    print(f"note: token belongs to {WHO}, config says {HF_USER}; using {WHO}")
    HF_USER = WHO
TUNED_REPO = f"{HF_USER}/{MODEL_NAME}"
GGUF_REPO  = f"{TUNED_REPO}-GGUF"
CACHE_REPO = f"{TUNED_REPO}-ckpt"
print(f"model {TUNED_REPO}\ngguf  {GGUF_REPO}\ncache {CACHE_REPO} (private)")

# --- ONE-TIME WIPE -------------------------------------------------------------------
# Uncomment for a single run to start completely clean, then COMMENT IT BACK OUT —
# otherwise every session deletes its own progress and resume never works.
#
# from huggingface_hub import delete_repo
# try:
#     delete_repo(CACHE_REPO, repo_type="model", token=HF_TOKEN)
#     print("old cache repo deleted")
# except Exception as e:
#     print(f"nothing to delete ({type(e).__name__})")
# -------------------------------------------------------------------------------------

create_repo(CACHE_REPO, repo_type="model", private=True, exist_ok=True)

def load_cached(name):
    try:
        p = hf_hub_download(CACHE_REPO, name, repo_type="model", token=HF_TOKEN)
        print(f"  {name}: found")
        return json.load(open(p))
    except Exception:
        print(f"  {name}: not cached")
        return None

def save_cached(name, obj):
    path = f"{SAVE}/{name}"
    json.dump(obj, open(path, "w"), indent=2, ensure_ascii=False)
    upload_file(path_or_fileobj=path, path_in_repo=name, repo_id=CACHE_REPO,
                repo_type="model", commit_message=f"cache {name}")
    print(f"  {name}: cached")

print("\ncache:")
TRAIN_DONE = load_cached("training_done.json") is not None
EVAL_BASE  = load_cached(f"eval_base_{BENCH_N}.json")
EVAL_TUNED = load_cached(f"eval_tuned_{BENCH_N}.json")

RESUME_FROM, START_STEP = None, 0
if not TRAIN_DONE:
    try:
        snap = snapshot_download(CACHE_REPO, repo_type="model", local_dir=CKPT_LOCAL,
                                 allow_patterns=["checkpoint/*"], token=HF_TOKEN)
        cdir = os.path.join(snap, "checkpoint")
        if os.path.isfile(os.path.join(cdir, "trainer_state.json")):
            st = json.load(open(os.path.join(cdir, "trainer_state.json")))
            ckpt_max = int(st.get("max_steps", 0))
            if ckpt_max and ckpt_max != TOTAL_STEPS:
                raise RuntimeError(
                    f"checkpoint was written for max_steps={ckpt_max:,} but this notebook "
                    f"is configured for {TOTAL_STEPS:,}. Resuming would put the cosine "
                    f"schedule on the wrong horizon. Set TARGET_EXAMPLES back to "
                    f"{ckpt_max*BATCH*ACCUM:,}, or use the ONE-TIME WIPE above.")
            START_STEP, RESUME_FROM = int(st.get("global_step", 0)), cdir
            if START_STEP >= TOTAL_STEPS:
                TRAIN_DONE = True
                print(f"\ncheckpoint is already at the target ({START_STEP:,})")
            else:
                print(f"\nresuming from step {START_STEP:,} / {TOTAL_STEPS:,} "
                      f"({START_STEP*BATCH*ACCUM:,} of {TARGET_EXAMPLES:,} examples seen)")
    except RuntimeError:
        raise
    except Exception as e:
        print(f"\nno checkpoint yet ({type(e).__name__}) — this is session 1")
else:
    print("\ntraining already finished; this session evaluates and publishes")
print(f"TRAIN_DONE = {TRAIN_DONE}")

## 5. Benchmarks, validated before anything expensive

Each dataset is shaped differently, so columns are detected rather than assumed. An
earlier version of this notebook looked only for a `question` column; `commonsenseqa-bn`
and `openbookqa-bn` both use `question_stem`, so both produced zero items, were skipped
with a one-line warning, and the "average across five benchmarks" was quietly an average
across three. The detector now covers both and this cell **asserts** that all five parse.

Three of the five have fewer than 1,000 rows in their test split, so splits are
concatenated to reach the target and the card reports exactly which were used.

In [ ]:
def _first(r, keys):
    for k in keys:
        if k in r and r[k] is not None and str(r[k]).strip() != "":
            return r[k]
    return None

Q_KEYS = ("question", "question_stem", "query", "stem", "goal", "prompt", "text")
A_KEYS = ("answer", "answerKey", "answer_key", "label", "gold", "correct")

def adapt_mmlu(r):
    """MMLU / CommonsenseQA / OpenBookQA: a choice list plus a key, an index or the text."""
    opts = r.get("choices") if r.get("choices") is not None else r.get("options")
    if isinstance(opts, dict):                      # {"label": [...], "text": [...]}
        opts = list(opts.get("text") or opts.get("label") or [])
    if not opts or len(opts) < 2:
        return None
    opts = [str(o) for o in opts]
    a = _first(r, A_KEYS)
    if a is None:
        return None
    if isinstance(a, str) and len(a.strip()) == 1 and a.strip().upper() in "ABCDE":
        gi = "ABCDE".index(a.strip().upper())
    elif isinstance(a, (int, np.integer)) and not isinstance(a, bool):
        gi = int(a)
    elif str(a).strip().isdigit():
        gi = int(str(a).strip())
    else:
        gi = next((i for i, o in enumerate(opts) if o.strip() == str(a).strip()), None)
    q = _first(r, Q_KEYS)
    if gi is None or q is None or gi < 0 or gi >= len(opts):
        return None
    return {"q": str(q), "ctx": "", "options": opts, "gold": gi}

def adapt_piqa(r):
    if not (r.get("sol1") and r.get("sol2")):
        return None
    q = _first(r, Q_KEYS)
    if q is None:
        return None
    return {"q": str(q), "ctx": "",
            "options": [str(r["sol1"]), str(r["sol2"])], "gold": int(r.get("label", 0))}

def adapt_boolq(r):
    """boolq_bn stores a boolean or a yes/no string, which no generic detector maps."""
    a = r.get("answer")
    if a is None:
        a = r.get("answer_bn")
    if isinstance(a, (bool, np.bool_)):
        gi = 0 if a else 1
    else:
        t = str(a).strip().lower()
        gi = 0 if t in ("true", "yes", "1", "হ্যাঁ", "হ্যা") else (
             1 if t in ("false", "no", "0", "না") else None)
    q = r.get("question")
    if gi is None or not q:
        return None
    return {"q": str(q), "ctx": str(r.get("passage") or ""),
            "options": ["হ্যাঁ", "না"], "gold": gi}

BENCHMARKS = [
    ("bangla-mmlu",      "hishab/bangla-mmlu",      adapt_mmlu, 0.25),
    ("commonsenseqa-bn", "hishab/commonsenseqa-bn", adapt_mmlu, 0.20),
    ("openbookqa-bn",    "hishab/openbookqa-bn",    adapt_mmlu, 0.25),
    ("piqa-bn",          "hishab/piqa-bn",          adapt_piqa, 0.50),
    ("boolq-bn",         "hishab/boolq_bn",         adapt_boolq, 0.50),
]

def rows_of(repo, need):
    parts, used, have = [], [], 0
    for split in ("test", "validation", "train"):
        if have >= need:
            break
        try:
            d = load_dataset(repo, split=split)
        except Exception:
            continue
        parts.append(d); used.append(f"{split}({len(d):,})"); have += len(d)
    if not parts:
        return None, None
    ds = parts[0] if len(parts) == 1 else concatenate_datasets(parts)
    return ds.shuffle(seed=SEED), " + ".join(used)

print("=" * 78)
print(f"PRE-FLIGHT: five benchmarks, target {BENCH_N:,} items each")
print("=" * 78)

BENCH, CHANCE, SPLITS, FAIL = {}, {}, {}, []
for name, repo, adapt, chance in BENCHMARKS:
    ds_b, used = rows_of(repo, BENCH_N)
    if ds_b is None:
        FAIL.append(name); print(f"\n{name:18} LOAD FAILED"); continue
    items, scanned = [], 0
    for r in ds_b:
        scanned += 1
        it = adapt(r)
        if it:
            items.append(it)
        if len(items) >= BENCH_N:
            break
    if not items:
        FAIL.append(name)
        print(f"\n{name:18} ADAPTER PRODUCED NOTHING, cols={ds_b.column_names}")
        print(f"  raw[0]: { {k: str(v)[:52] for k, v in ds_b[0].items()} }")
        continue
    BENCH[name], CHANCE[name], SPLITS[name] = items, chance, used
    ex = items[0]
    print(f"\n{name:18} {len(items):,} items from {used}")
    print(f"  {len(ex['options'])} options | chance {chance:.2f} | "
          f"parsed {len(items)}/{scanned} scanned")
    print(f"  q    : {ex['q'][:70]}")
    print(f"  gold : {ex['gold']} -> {ex['options'][ex['gold']][:44]}")

assert not FAIL, (
    f"{len(FAIL)} benchmark(s) failed to parse: {FAIL}. Fix the adapter before training — "
    "otherwise the card will claim five benchmarks while reporting fewer.")
TOTAL_OPTS = sum(len(i["options"]) for v in BENCH.values() for i in v)
SE = math.sqrt(2) * 0.5 / math.sqrt(BENCH_N)
print(f"\nall five ready | {TOTAL_OPTS:,} option scores per model")
print(f"noise threshold for a per-benchmark difference: {2*SE:.3f}")

## 6. Scoring

`score_packed` sorts options by sequence length, packs them into nearly length-homogeneous
chunks under a token budget, and takes the log-softmax only at the few positions that
score option tokens — never over the whole 262k-token vocabulary at every position, which
is what would exhaust VRAM.

Gemma-3 runs in fp32 on a T4 (fp16 is numerically unsafe for this architecture) and its
sliding-window attention goes through xformers, whose accumulation order depends on batch
shape. So batching shifts a mean log-probability by order 1e-2, about 1%. That is
harmless for argmax but it means a tight tolerance on raw scores is the wrong check.
`validate_scorer` therefore measures the score drift *and* asserts on **decision
agreement**, then picks the fastest mode that reproduces the unbatched decisions exactly.
If packing disagrees it falls back to one forward per question, which is still 4–5× faster
than one forward per option.

In [ ]:
def build_prompt_ids(tok, item):
    user = f"{item['ctx']}\n\n{item['q']}" if item["ctx"] else item["q"]
    p = tok.apply_chat_template(to_chat(user), tokenize=False, add_generation_prompt=True)
    return tok(p, add_special_tokens=False)["input_ids"][-(MAX_SEQ - 64):]

def pairs_for(tok, items):
    pairs, spans = [], []
    for it in items:
        pid = build_prompt_ids(tok, it)
        a = len(pairs)
        for o in it["options"]:
            oid = tok(o, add_special_tokens=False)["input_ids"]
            pairs.append((pid, oid if oid else [tok.eos_token_id or 0]))
        spans.append((a, len(pairs)))
    return pairs, spans

@torch.no_grad()
def _forward_scores(model, tok, idxs, pairs, out):
    pad = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)
    seqs = [pairs[i][0] + pairs[i][1] for i in idxs]
    L = max(len(s) for s in seqs)
    inp = torch.full((len(seqs), L), pad, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    for k, s in enumerate(seqs):
        inp[k, :len(s)] = torch.tensor(s, dtype=torch.long)
        att[k, :len(s)] = 1
    logits = model(input_ids=inp.cuda(), attention_mask=att.cuda()).logits
    for k, i in enumerate(idxs):
        p, o = pairs[i]
        if not o:
            continue
        # position len(p)-1 predicts o[0], len(p) predicts o[1], and so on
        sl = logits[k, len(p) - 1: len(p) - 1 + len(o), :].float().log_softmax(-1)
        tgt = torch.tensor(o, device=sl.device).unsqueeze(1)
        out[i] = float(sl.gather(1, tgt).sum()) / len(o)
    del logits

@torch.no_grad()
def score_packed(model, tok, pairs, spans=None, tok_budget=TOK_BUDGET, max_slack=1.08):
    out = [-1e9] * len(pairs)
    order = sorted(range(len(pairs)), key=lambda i: len(pairs[i][0]) + len(pairs[i][1]))
    buf, lo, hi = [], None, None
    for i in order:
        n = len(pairs[i][0]) + len(pairs[i][1])
        nlo = n if lo is None else min(lo, n)
        nhi = n if hi is None else max(hi, n)
        if buf and (nhi > max_slack * nlo + 8 or nhi * (len(buf) + 1) > tok_budget):
            _forward_scores(model, tok, buf, pairs, out)
            buf, lo, hi = [], None, None
            nlo = nhi = n
        buf.append(i); lo, hi = nlo, nhi
    if buf:
        _forward_scores(model, tok, buf, pairs, out)
    return out

@torch.no_grad()
def score_per_item(model, tok, pairs, spans, **kw):
    """One forward per question. Options of an item share a prompt, so the batch is 2–5
    rows with only a few tokens of padding and the kernel path barely varies."""
    out = [-1e9] * len(pairs)
    for a, b in spans:
        _forward_scores(model, tok, list(range(a, b)), pairs, out)
    return out

@torch.no_grad()
def score_one(model, tok, prompt_ids, option_ids):
    """Reference: batch of one, no padding. Only used to validate the others."""
    ids = torch.tensor([prompt_ids + option_ids], device="cuda")
    logits = model(input_ids=ids).logits
    sl = logits[0, len(prompt_ids) - 1: len(prompt_ids) - 1 + len(option_ids), :] \
        .float().log_softmax(-1)
    tgt = torch.tensor(option_ids, device=sl.device).unsqueeze(1)
    return float(sl.gather(1, tgt).sum()) / len(option_ids)

SCORERS = {"packed": score_packed, "per_item": score_per_item}

@torch.no_grad()
def validate_scorer(model, tok, n_mmlu=24, n_boolq=8):
    items = BENCH["bangla-mmlu"][:n_mmlu] + BENCH["boolq-bn"][:n_boolq]
    pairs, spans = pairs_for(tok, items)
    ref = [score_one(model, tok, p, o) for p, o in pairs]
    ref_dec = [int(np.argmax(ref[a:b])) for a, b in spans]

    report = {}
    for mode in ("packed", "per_item"):
        got = SCORERS[mode](model, tok, pairs, spans)
        dec = [int(np.argmax(got[a:b])) for a, b in spans]
        d = [abs(x - y) for x, y in zip(got, ref)]
        agree = sum(int(x == y) for x, y in zip(dec, ref_dec)) / len(dec)
        report[mode] = {"max_abs": round(max(d), 5),
                        "median_abs": round(float(np.median(d)), 5),
                        "decision_agreement": round(agree, 4),
                        "items": len(dec), "options": len(pairs)}
        print(f"  {mode:9} max|Δ| {max(d):.2e}  median|Δ| {np.median(d):.2e}  "
              f"decisions agree {agree:.1%} over {len(dec)} items")
    for mode in ("packed", "per_item"):
        if report[mode]["decision_agreement"] == 1.0:
            print(f"  -> scoring with '{mode}'")
            return mode, report
    raise AssertionError(
        f"neither batched mode reproduces the unbatched decisions — {report}. "
        "Do not trust the numbers; lower TOK_BUDGET and re-run.")

@torch.no_grad()
def run_benchmarks(model, tok, tag, mode):
    model.eval()
    results, per_bench_time = {}, {}
    for name, items in BENCH.items():
        t0 = time.time()
        pairs, spans = pairs_for(tok, items)
        scores = SCORERS[mode](model, tok, pairs, spans)
        correct = sum(int(int(np.argmax(scores[a:b])) == it["gold"])
                      for (a, b), it in zip(spans, items))
        acc = correct / len(items)
        results[name] = round(acc, 4)
        per_bench_time[name] = round(time.time() - t0, 1)
        print(f"  {tag} {name:18} {acc:.4f}   ({per_bench_time[name]:.0f}s, "
              f"{len(pairs):,} options, {mode})")
    results["average"] = round(float(np.mean([v for k, v in results.items()
                                              if k != "average"])), 4)
    print(f"  {tag} {'AVERAGE':18} {results['average']:.4f}")
    return results, per_bench_time

@torch.no_grad()
def perplexity(model, tok, texts):
    model.eval()
    nll, ntok = 0.0, 0
    for t in texts:
        ids = tok(t, add_special_tokens=False, truncation=True,
                  max_length=MAX_SEQ)["input_ids"]
        if len(ids) < 16:
            continue
        x = torch.tensor([ids], device="cuda")
        nll += float(model(input_ids=x, labels=x).loss) * (len(ids) - 1)
        ntok += len(ids) - 1
    return round(math.exp(nll / max(ntok, 1)), 2)

@torch.no_grad()
def generate(model, tok, texts, max_new=220):
    model.eval()
    outs = []
    for t in texts:
        p = tok.apply_chat_template(to_chat(t), tokenize=False, add_generation_prompt=True)
        enc = tok(p, return_tensors="pt", add_special_tokens=False).to("cuda")
        g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        outs.append(tok.decode(g[0][enc["input_ids"].shape[1]:],
                               skip_special_tokens=True).strip())
    return outs

def load_model(repo, for_training=False):
    clear()
    m, t = FastModel.from_pretrained(model_name=repo, max_seq_length=MAX_SEQ,
                                     dtype=None, load_in_4bit=False,
                                     full_finetuning=for_training)
    t = get_chat_template(t, chat_template="gemma-3")
    if not for_training:
        m.eval()
    return m, t

def drop(m, t):
    del m, t
    clear()

SCORER_CHECK = {}

## 7. Held-out perplexity corpus

Shuffled `Bangla-Instruct` rows 200,000–200,800 — far past the 115,000 the training slice
is drawn from, so these have certainly not been seen.

In [ ]:
_ppl_raw = load_dataset(SFT_DATASET, split="train").shuffle(seed=SEED)
DATASET_TOTAL = len(_ppl_raw)
assert PPL_FROM > SELECT_N, "perplexity rows overlap the training slice"
PPL_TEXTS = [str(r["response"]).strip()
             for r in _ppl_raw.select(range(PPL_FROM, PPL_TO))
             if len(str(r["response"]).split()) >= 25][:PPL_N]
del _ppl_raw; gc.collect()
print(f"{len(PPL_TEXTS)} held-out responses, median "
      f"{int(np.median([len(t.split()) for t in PPL_TEXTS]))} words "
      f"| dataset has {DATASET_TOTAL:,} rows")
assert len(PPL_TEXTS) >= 50, "not enough held-out text for a stable perplexity"

## 8. Training data

`Bangla-Instruct` is **342,391** instruction-response pairs generated natively in Bangla
rather than machine-translated. (An earlier draft of this notebook said "100K"; that was
wrong.) The translated Alpaca derivatives most Bangla fine-tunes use inherit English
sentence structure and translation artefacts.

Three different numbers are reported below, because conflating them is how a card ends up
overstating its training: how many of **all 342,391** rows fit in 512 tokens, how many of
the **115,000-row slice** survive both filters, and how many the model **actually sees**.

In [ ]:
if not TRAIN_DONE:
    raw = load_dataset(SFT_DATASET, split="train")
    print(raw)
    cols = raw.column_names
    INS_COL = next(c for c in ["instruction", "prompt", "question", "input_text"]
                   if c in cols)
    OUT_COL = next(c for c in ["response", "output", "answer", "completion"] if c in cols)
    CTX_COL = next((c for c in ["input", "context"] if c in cols), None)
    print(f"instruction: {INS_COL} | response: {OUT_COL} | extra input: {CTX_COL}")
    raw = raw.shuffle(seed=SEED)

    def build(ex):
        user = str(ex[INS_COL]).strip()
        if CTX_COL and str(ex.get(CTX_COL) or "").strip():
            user += "\n\n" + str(ex[CTX_COL]).strip()
        return {"text": tokenizer.apply_chat_template(
            to_chat(user, str(ex[OUT_COL]).strip()), tokenize=False)}

    def ntok(t):
        return len(tokenizer(t, add_special_tokens=False)["input_ids"])
else:
    print("training finished — skipping data preparation")

### 8a. Load the model for training

The tokenizer is needed to build the dataset, so the model is loaded before the census.

In [ ]:
if not TRAIN_DONE:
    clear()
    model, tokenizer = load_model(BASE_MODEL, for_training=True)
    N_PARAMS = sum(p.numel() for p in model.parameters())
    TRAINABLE = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{N_PARAMS/1e9:.2f}B params | trainable {TRAINABLE/1e9:.2f}B "
          f"({100*TRAINABLE/N_PARAMS:.1f}%) | {vram():.2f} GB")
    assert TRAINABLE > 0.9 * N_PARAMS, \
        "Not a full fine-tune — full_finetuning=True did not take effect."
else:
    N_PARAMS, TRAINABLE = 999_885_952, 999_885_952

### 8b. Census over the whole dataset

Templating 342,391 rows one at a time is slow, so the chat template's overhead is measured
once on 500 real rows and added to batched token counts. The cell prints the worst-case
error of that shortcut so you can see it is a rounding detail. Some rows carry `None`,
which the fast tokenizer rejects outright, so every value is coerced to a string.

In [ ]:
if not TRAIN_DONE and FULL_CENSUS:
    t0 = time.time()

    def _txt(v):
        return "" if v is None else str(v)

    probe = raw.select(range(500))
    exact = [ntok(build(ex)["text"]) for ex in probe]
    parts = [ntok(_txt(ex[INS_COL])) + ntok(_txt(ex[OUT_COL])) for ex in probe]
    diffs = [e - p for e, p in zip(exact, parts)]
    OVERHEAD = int(np.median(diffs))
    print(f"chat-template overhead: {OVERHEAD} tokens "
          f"(worst-case error {max(abs(d - OVERHEAD) for d in diffs)} tokens)")

    def _len_batch(b):
        ins = [_txt(v) for v in b[INS_COL]]
        out = [_txt(v) for v in b[OUT_COL]]
        li = tokenizer(ins, add_special_tokens=False)["input_ids"]
        lo = tokenizer(out, add_special_tokens=False)["input_ids"]
        return {"n": [len(a) + len(c) + OVERHEAD for a, c in zip(li, lo)],
                "empty": [1 if (not a.strip() or not c.strip()) else 0
                          for a, c in zip(ins, out)]}

    census = raw.map(_len_batch, batched=True, batch_size=1000, num_proc=2,
                     remove_columns=raw.column_names, desc="token census")
    NS = np.array(census["n"])
    N_EMPTY = int(np.array(census["empty"]).sum())
    del census
    FULL_FIT = int((NS <= MAX_SEQ).sum())
    print(f"\n--- whole dataset, {DATASET_TOTAL:,} rows ---")
    print(f"  empty instruction or response : {N_EMPTY:,}")
    print(f"  median {int(np.median(NS))} tokens | 90th {int(np.percentile(NS,90))} "
          f"| max {int(NS.max())}")
    print(f"  fit in {MAX_SEQ} tokens : {FULL_FIT:,} ({100*FULL_FIT/DATASET_TOTAL:.1f}%)")
    print(f"  dropped                : {DATASET_TOTAL-FULL_FIT:,}")
    print(f"  one full epoch would be {math.ceil(FULL_FIT/(BATCH*ACCUM)):,} steps "
          f"({FULL_FIT/TARGET_EXAMPLES:.1f}x this run's target)")
    print(f"  census took {(time.time()-t0)/60:.1f} min")
    del NS; gc.collect()
else:
    FULL_FIT, N_EMPTY = None, 0
    if not TRAIN_DONE:
        print("FULL_CENSUS disabled")

### 8c. The slice that is actually used

In [ ]:
if not TRAIN_DONE:
    sel = raw.select(range(min(SELECT_N, DATASET_TOTAL)))
    pulled = len(sel)
    sel = sel.filter(lambda e: str(e[INS_COL] or "").strip()
                           and str(e[OUT_COL] or "").strip(),
                     num_proc=2, desc="drop empty rows")
    nonempty = len(sel)

    ds = sel.map(build, remove_columns=raw.column_names, num_proc=2, desc="templating")
    ds = ds.filter(lambda e: ntok(e["text"]) <= MAX_SEQ, num_proc=2,
                   desc="512-token filter")
    POOL = len(ds)

    print(f"\n--- selected slice ---")
    print(f"  pulled from shuffled dataset : {pulled:,}")
    print(f"  after dropping empty rows    : {nonempty:,}  (-{pulled-nonempty:,})")
    print(f"  fit in {MAX_SEQ} tokens              : {POOL:,}  "
          f"({100*POOL/nonempty:.1f}% of non-empty, -{nonempty-POOL:,})")

    assert POOL >= TARGET_EXAMPLES, (
        f"only {POOL:,} examples survived but {TARGET_EXAMPLES:,} are needed — "
        f"raise SELECT_N (currently {SELECT_N:,}).")

    ds = ds.select(range(TARGET_EXAMPLES))     # exactly one epoch, no repeats
    print(f"  kept for training            : {len(ds):,}  <- each seen once")
    print(f"  left on the table            : {POOL-TARGET_EXAMPLES:,}")

    lens = [ntok(t) for t in ds["text"][:2000]]
    print(f"\ntraining set token length: median {int(np.median(lens))}, "
          f"90th {int(np.percentile(lens,90))}, max {max(lens)}")
    print("\n--- formatted example ---\n", ds[0]["text"][:800])
else:
    POOL = TARGET_EXAMPLES

## 9. Trainer

Loss on the assistant turn only. `max_grad_norm=1.0` and a low learning rate matter more
here than in LoRA: every weight moves, so a bad step damages the multilingual ability the
base model arrived with.

`ignore_data_skip=True` is what makes resuming cheap — without it Trainer replays the
dataloader from row zero to catch up, which at step 3,000 means re-reading 48,000 examples
before doing any work.

`group_by_length=True` batches examples of similar length together. Every batch is padded
to its longest member, so mixing a 90-token example with a 500-token one wastes four
fifths of the short one's compute. Grouping buys 15–30% and costs nothing in quality.

The callback does three jobs: it stops before Kaggle does, it **uploads each checkpoint
as Trainer writes it**, and it keeps the disk clear. The disk part is not housekeeping —
Trainer rotates old checkpoints only *after* writing the new one, so two ~4.6 GB
checkpoints coexist at the moment of writing, and in a 20 GB quota that is how the
optimizer save runs out of room. The callback sweeps already-uploaded checkpoints one step
**before** each save.

In [ ]:
if not TRAIN_DONE:
    try:
        from trl import SFTConfig
        CFG = SFTConfig
    except ImportError:
        from transformers import TrainingArguments as CFG
    _P = set(inspect.signature(CFG.__init__).parameters)

    wanted = dict(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        warmup_steps=100, max_steps=TOTAL_STEPS, learning_rate=LR,
        fp16=not BF16, bf16=BF16, logging_steps=20, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="cosine", max_grad_norm=1.0,
        seed=SEED, output_dir=OUTDIR, report_to="none",
        save_strategy="steps", save_steps=CKPT_EVERY, save_total_limit=1,
        ignore_data_skip=True, gradient_checkpointing=True, group_by_length=True,
        max_seq_length=MAX_SEQ, dataset_text_field="text",
        dataset_num_proc=2, packing=False,
    )
    kw = {k: v for k, v in wanted.items() if k in _P}
    extra = {k: wanted[k] for k in ["max_seq_length", "dataset_text_field",
                                    "dataset_num_proc", "packing"] if k not in _P}
    dropped = sorted(set(wanted) - set(kw) - set(extra))
    if dropped:
        print("not supported by this version, skipped:", dropped)

    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
                         args=CFG(**kw), **extra)
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<start_of_turn>user\n",
        response_part="<start_of_turn>model\n")

    lab = np.array(trainer.train_dataset[0]["labels"])
    sup = int((lab != -100).sum())
    print(f"\n{sup}/{len(lab)} tokens supervised ({sup/len(lab):.1%})")
    print("supervised begins:",
          repr(tokenizer.decode([t for t in lab if t != -100][:40])))
    assert 0 < sup < len(lab), "response-only masking failed — check the turn markers."

In [ ]:
def complete_checkpoints(d):
    """Only dirs with trainer_state.json AND optimizer state can be resumed from. A
    weights-only directory would resume silently from step 0 with a fresh optimizer, which
    looks like it worked and quietly throws the run away."""
    out = []
    for p in glob.glob(f"{d}/checkpoint-*"):
        if not os.path.isdir(p):
            continue
        if (os.path.isfile(os.path.join(p, "trainer_state.json"))
                and any(f.startswith("optimizer") for f in os.listdir(p))):
            out.append(p)
    return sorted(out, key=lambda p: int(p.rsplit("-", 1)[-1]))

class BudgetStop(TrainerCallback):
    def __init__(self, budget_min, probe_at=20):
        self.budget = budget_min * 60
        self.probe_at, self.probed = probe_at, False
        self.t0, self.start_step = None, 0
        self.pushed, self.last_pushed = 0, 0

    def on_train_begin(self, args, state, control, **kw):
        self.t0, self.start_step = time.time(), state.global_step
        self.last_pushed = state.global_step
        if RESUME_FROM and os.path.isdir(CKPT_LOCAL):
            shutil.rmtree(CKPT_LOCAL, ignore_errors=True)
            print(f"[disk] removed the resume copy; "
                  f"{shutil.disk_usage(WORK).free/1e9:.1f} GB free in {WORK}")

    def _sweep(self, args, keep_after):
        freed = 0
        for d in glob.glob(f"{args.output_dir}/checkpoint-*"):
            if not os.path.isdir(d):
                continue
            try:
                step = int(d.rsplit("-", 1)[-1])
            except ValueError:
                continue
            complete = os.path.isfile(os.path.join(d, "trainer_state.json"))
            if step <= keep_after or not complete:
                freed += sum(os.path.getsize(os.path.join(dp, f))
                             for dp, _, fs in os.walk(d) for f in fs)
                shutil.rmtree(d, ignore_errors=True)
        return freed / 1e9

    def on_save(self, args, state, control, **kw):
        """Runs inside training. This is the only thing standing between an aborted
        session and starting over."""
        try:
            ck = complete_checkpoints(args.output_dir)
            if not ck:
                print("[autosave] no complete checkpoint on disk yet, skipping")
                return control
            t = time.time()
            upload_folder(repo_id=CACHE_REPO, repo_type="model", folder_path=ck[-1],
                          path_in_repo="checkpoint", delete_patterns="*",
                          commit_message=f"autosave step {state.global_step}")
            self.pushed += 1
            self.last_pushed = int(ck[-1].rsplit("-", 1)[-1])
            freed = self._sweep(args, keep_after=self.last_pushed - 1)
            print(f"[autosave] step {state.global_step:,}/{TOTAL_STEPS:,} -> "
                  f"{CACHE_REPO}  ({(time.time()-t)/60:.1f} min, push #{self.pushed}) | "
                  f"freed {freed:.1f} GB, {shutil.disk_usage(WORK).free/1e9:.1f} GB free")
        except Exception as e:
            print(f"[autosave] FAILED, training continues: {type(e).__name__}: "
                  f"{str(e)[:140]}")
        return control

    def on_step_end(self, args, state, control, **kw):
        done = state.global_step - self.start_step
        el = time.time() - self.t0

        if args.save_steps and (state.global_step + 1) % args.save_steps == 0:
            freed = self._sweep(args, keep_after=self.last_pushed)
            free_gb = shutil.disk_usage(WORK).free / 1e9
            if freed or free_gb < 8:
                print(f"[disk] pre-save sweep freed {freed:.1f} GB; "
                      f"{free_gb:.1f} GB free in {WORK}")
            if free_gb < 6:
                print(f"[disk] WARNING: only {free_gb:.1f} GB free — "
                      "the next checkpoint needs ~5 GB")

        if not self.probed and done >= self.probe_at:
            self.probed = True
            sps = el / done
            rem = TOTAL_STEPS - state.global_step
            fits = int(self.budget / sps)
            print(f"\n[probe] {sps:.1f} s/step | {rem:,} steps left "
                  f"= {rem*sps/3600:.1f} h | budget {self.budget/3600:.1f} h")
            if rem > fits:
                print(f"[probe] ~{fits:,} steps fit this session; about "
                      f"{math.ceil(rem/max(fits,1))} more session(s) after this one")
            else:
                print("[probe] this session should finish training")

        if el > self.budget:
            control.should_training_stop = True
            control.should_save = True
            print(f"\n[budget] {self.budget/60:.0f} min reached — stopping at step "
                  f"{state.global_step:,} and checkpointing")
        return control

## 10. Train

In [ ]:
if not TRAIN_DONE:
    for _n in ["probe", "exact", "parts", "diffs", "sel", "lens"]:
        if _n in globals():
            del globals()[_n]
    gc.collect()

    for _d in glob.glob(f"{OUTDIR}/checkpoint-*"):
        if os.path.isdir(_d) and not os.path.isfile(
                os.path.join(_d, "trainer_state.json")):
            print(f"removing incomplete {_d}")
            shutil.rmtree(_d, ignore_errors=True)

    free = shutil.disk_usage(WORK).free / 1e9
    print(f"disk free before training: {free:.1f} GB in {WORK}")
    print(f"scratch free: {shutil.disk_usage(SCRATCH).free/1e9:.1f} GB in {SCRATCH}")
    assert free > 7, (
        f"only {free:.1f} GB free in {WORK}; a checkpoint needs ~5 GB.")

    TRAIN_BUDGET_MIN = max(20, SESSION_LIMIT_MIN - elapsed_min() - RESERVE_MIN)
    print(f"training budget this session: {TRAIN_BUDGET_MIN:.0f} min "
          f"(reserving {RESERVE_MIN} min for evaluation and publishing)")
    trainer.add_callback(BudgetStop(TRAIN_BUDGET_MIN))   # without this nothing above runs

    clear()
    t0 = time.time()
    stats = trainer.train(resume_from_checkpoint=RESUME_FROM)
    TRAIN_MIN_SESSION = (time.time() - t0) / 60
    VRAM_TRAIN = vram()
    END_STEP = trainer.state.global_step
    REACHED_TARGET = END_STEP >= TOTAL_STEPS
    CURVE = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
    print(f"\nsession: {TRAIN_MIN_SESSION:.0f} min | steps {START_STEP:,} -> {END_STEP:,} "
          f"of {TOTAL_STEPS:,} | peak VRAM {VRAM_TRAIN:.2f} GB")
    print(f"examples seen: {END_STEP*BATCH*ACCUM:,} / {TARGET_EXAMPLES:,}")
    if CURVE:
        print(f"last logged loss: {CURVE[-1][1]:.4f}")
        assert not np.isnan(CURVE[-1][1]), "loss is NaN — lower the learning rate"
    print("REACHED_TARGET =", REACHED_TARGET)
else:
    REACHED_TARGET, VRAM_TRAIN, CURVE = True, None, None

## 11. Either checkpoint and stop, or bank the weights and continue

If training finished, the weights are saved and pushed **before** evaluation runs, so a
failure later in the notebook cannot cost them. Every cell after this is guarded, so Run
All simply falls through when the run is unfinished.

In [ ]:
if not TRAIN_DONE and not REACHED_TARGET:
    ck = complete_checkpoints(OUTDIR)
    if not ck or int(ck[-1].rsplit("-", 1)[-1]) < END_STEP:
        try:
            trainer._save_checkpoint(model=None, trial=None)   # private, version-dependent
        except Exception as e:
            print(f"forced save failed ({type(e).__name__}); "
                  "falling back to the last periodic checkpoint")
        ck = complete_checkpoints(OUTDIR)
    assert ck, ("No resumable checkpoint on disk. Lower CKPT_EVERY and re-run — the "
                "session ended before the first periodic save.")
    latest = ck[-1]
    CKPT_STEP = int(json.load(open(os.path.join(latest, "trainer_state.json")))
                    ["global_step"])
    size = sum(os.path.getsize(os.path.join(dp, f))
               for dp, _, fs in os.walk(latest) for f in fs) / 1e9
    print(f"uploading {latest} ({size:.1f} GB) to {CACHE_REPO} ...")
    upload_folder(repo_id=CACHE_REPO, repo_type="model", folder_path=latest,
                  path_in_repo="checkpoint", delete_patterns="*",
                  commit_message=f"checkpoint at step {CKPT_STEP}/{TOTAL_STEPS}")
    rem = TOTAL_STEPS - CKPT_STEP
    sps = TRAIN_MIN_SESSION * 60 / max(END_STEP - START_STEP, 1)
    print(f"""
================================================================================
  SESSION DONE, RUN NOT FINISHED

  progress   {CKPT_STEP:,} / {TOTAL_STEPS:,} steps
             {CKPT_STEP*BATCH*ACCUM:,} / {TARGET_EXAMPLES:,} examples
             {100*CKPT_STEP/TOTAL_STEPS:.0f}% complete
  rate       {sps:.1f} s/step this session
  remaining  {rem:,} steps (~{rem*sps/3600:.1f} h)

  Next: Save & Run All again. The notebook finds the checkpoint, skips the data
  census, and continues from step {CKPT_STEP:,}. Nothing to edit.

  Evaluation, GGUF, the card and the push all happen automatically on the
  session that reaches step {TOTAL_STEPS:,}.
================================================================================
""")

elif not TRAIN_DONE and REACHED_TARGET:
    print("training complete — banking the weights before evaluation")
    model.save_pretrained(SAVE); tokenizer.save_pretrained(SAVE)
    try:
        model.save_pretrained_merged(MERGED, tokenizer, save_method="merged_16bit")
    except Exception:
        shutil.copytree(SAVE, MERGED, dirs_exist_ok=True)   # already merged
    create_repo(TUNED_REPO, repo_type="model", private=False, exist_ok=True)
    upload_folder(repo_id=TUNED_REPO, folder_path=MERGED,
                  commit_message=f"Full fine-tune for Bangla, {TARGET_EXAMPLES:,} "
                                 f"examples over {TOTAL_STEPS:,} steps")
    save_cached("training_done.json", {
        "steps": TOTAL_STEPS, "examples": TARGET_EXAMPLES, "max_seq": MAX_SEQ,
        "lr": LR, "batch": BATCH, "accum": ACCUM, "peak_vram_gb": round(VRAM_TRAIN, 2),
        "pool_after_filter": POOL, "dataset_total": DATASET_TOTAL,
        "fit_in_maxseq_whole_dataset": FULL_FIT, "empty_rows": N_EMPTY,
        "hardware": GPU, "seed": SEED, "loss_curve": CURVE,
        "total_params": N_PARAMS, "trainable_params": TRAINABLE,
    })
    TRAIN_DONE = True
    drop(model, tokenizer)
    print(f"weights are safe at https://huggingface.co/{TUNED_REPO}")

if TRAIN_DONE:
    meta = load_cached("training_done.json") or {}
    VRAM_TRAIN = meta.get("peak_vram_gb") or VRAM_TRAIN
    CURVE = meta.get("loss_curve") or CURVE
    POOL = meta.get("pool_after_filter", POOL)
    FULL_FIT = meta.get("fit_in_maxseq_whole_dataset", FULL_FIT)
    N_PARAMS = meta.get("total_params", N_PARAMS)
    TRAINABLE = meta.get("trainable_params", TRAINABLE)
print(f"session elapsed: {elapsed_min():.0f} min")

## 12. Evaluate the fine-tune

Loaded fresh from the saved weights rather than reused from the training session, so the
numbers come from the same artifact users will download.

In [ ]:
if TRAIN_DONE and EVAL_TUNED is None:
    print(f"loading {SAVE}")
    model, tok = load_model(SAVE)

    print("\nvalidating the scorer on this model:")
    MODE_T, rep_t = validate_scorer(model, tok)

    BN = ["বাংলাদেশের রাজধানী ঢাকা একটি জনবহুল শহর।",
          "বিজ্ঞান ও প্রযুক্তির অগ্রগতি আমাদের জীবনযাত্রাকে সহজ করেছে।",
          "তিনি প্রতিদিন সকালে নদীর ধারে হাঁটতে যেতেন এবং পাখির ডাক শুনতেন।",
          "অর্থনৈতিক উন্নয়নের জন্য শিক্ষার বিকল্প নেই।"]
    tc = sum(len(tok(s, add_special_tokens=False)["input_ids"]) for s in BN)
    wc = sum(len(s.split()) for s in BN)
    en = "The capital of Bangladesh, Dhaka, is a densely populated city."
    en_tpw = len(tok(en, add_special_tokens=False)["input_ids"]) / len(en.split())
    tpw = tc / wc
    print(f"tokens/word: Bangla {tpw:.2f}, English {en_tpw:.2f} ({tpw/en_tpw:.1f}x)")

    print("\nfine-tuned model:")
    t_bench, t_time = run_benchmarks(model, tok, "tuned", MODE_T)
    t_ppl = perplexity(model, tok, PPL_TEXTS)
    print("  tuned perplexity:", t_ppl)
    t_gen = generate(model, tok, PROBES)
    EVAL_TUNED = {"bench": t_bench, "ppl": t_ppl, "gen": dict(zip(PROBES, t_gen)),
                  "time": t_time, "n": BENCH_N,
                  "tokens_per_word": {"bangla": tpw, "english": en_tpw},
                  "scorer": {"mode": MODE_T, "report": rep_t}}
    save_cached(f"eval_tuned_{BENCH_N}.json", EVAL_TUNED)
    drop(model, tok)
elif TRAIN_DONE:
    print("fine-tune results from cache")

if TRAIN_DONE:
    SCORER_CHECK["tuned"] = EVAL_TUNED.get("scorer", {})
    TUNED_BENCH = EVAL_TUNED["bench"]
    TUNED_PPL   = EVAL_TUNED["ppl"]
    TUNED_GEN   = [EVAL_TUNED["gen"][q] for q in PROBES]
    TPW   = EVAL_TUNED["tokens_per_word"]["bangla"]
    ENTPW = EVAL_TUNED["tokens_per_word"]["english"]
    print(f"session elapsed: {elapsed_min():.0f} min")

## 13. Evaluate the base model

The before-picture, measured on the same items with the same scorer. Without it there is
no Δ column.

In [ ]:
if TRAIN_DONE and EVAL_BASE is None:
    print(f"loading {BASE_MODEL}")
    model, tok = load_model(BASE_MODEL)

    print("\nvalidating the scorer on the base model:")
    MODE_B, rep_b = validate_scorer(model, tok)

    print("\nbase model:")
    b_bench, b_time = run_benchmarks(model, tok, "base", MODE_B)
    b_ppl = perplexity(model, tok, PPL_TEXTS)
    print("  base perplexity:", b_ppl)
    b_gen = generate(model, tok, PROBES)
    EVAL_BASE = {"bench": b_bench, "ppl": b_ppl, "gen": dict(zip(PROBES, b_gen)),
                 "time": b_time, "n": BENCH_N, "splits": SPLITS, "chance": CHANCE,
                 "scorer": {"mode": MODE_B, "report": rep_b}}
    save_cached(f"eval_base_{BENCH_N}.json", EVAL_BASE)
    drop(model, tok)
elif TRAIN_DONE:
    print("base results from cache")

if TRAIN_DONE:
    SCORER_CHECK["base"] = EVAL_BASE.get("scorer", {})
    BASE_BENCH = EVAL_BASE["bench"]
    BASE_PPL   = EVAL_BASE["ppl"]
    BASE_GEN   = [EVAL_BASE["gen"][q] for q in PROBES]

    NAMES = [k for k in BASE_BENCH if k != "average"]
    D_AVG = TUNED_BENCH["average"] - BASE_BENCH["average"]

    print(f"\n{'benchmark':20}{'chance':>8}{'base':>8}{'tuned':>8}{'delta':>9}{'sigma':>7}")
    for k in NAMES:
        d = TUNED_BENCH[k] - BASE_BENCH[k]
        print(f"{k:20}{CHANCE[k]:8.2f}{BASE_BENCH[k]:8.4f}{TUNED_BENCH[k]:8.4f}"
              f"{d:+9.4f}{abs(d)/SE:7.1f}")
    print(f"{'average':20}{'—':>8}{BASE_BENCH['average']:8.4f}"
          f"{TUNED_BENCH['average']:8.4f}{D_AVG:+9.4f}")
    print(f"{'perplexity':20}{'—':>8}{BASE_PPL:8.2f}{TUNED_PPL:8.2f}"
          f"{TUNED_PPL-BASE_PPL:+9.2f}   (lower is better)")
    print(f"\nSE of a per-benchmark difference at n={BENCH_N}: ±{SE:.3f}; "
          f"anything under {2*SE:.3f} is noise")

    EXAMPLES = [{"q": q, "base": b, "tuned": t}
                for q, b, t in zip(PROBES, BASE_GEN, TUNED_GEN)]
    print(f"session elapsed: {elapsed_min():.0f} min")

## 14. Figures published by other projects

Nothing here is measured. These are transcriptions, so the card can show them without
implying they were reproduced.

In [ ]:
# Source: hishab/titulm-llama-3.2-1b-v2.0 card, 0-shot rows. None = not reported.
TITULM = {
    "llama-3.2-1b (base)": {"bangla-mmlu": 0.29, "boolq-bn": 0.55,
                            "commonsenseqa-bn": 0.22, "openbookqa-bn": 0.33,
                            "piqa-bn": 0.53},
    "titulm-llama-3.2-1b-v2.0": {"bangla-mmlu": 0.25, "boolq-bn": None,
                                 "commonsenseqa-bn": 0.26, "openbookqa-bn": 0.32,
                                 "piqa-bn": 0.58},
}
# Source: md-nishat-008/TigerLLM-1B-it card, Table 2 of the TigerLLM paper (ACL 2025).
TIGER_COLS = ["MMLU-bn", "PangBench-bn", "BanglaQuaD", "mHumanEval-bn",
              "BEnQA", "BanglaRQA"]
TIGER_ROWS = [
    ("GPT-3.5",                             [0.55, 0.55, 0.50, 0.56, 0.50, 0.49]),
    ("Gemini 1.5 Flash",                    [0.66, 0.57, 0.62, 0.58, 0.56, 0.61]),
    ("GPT-4o-mini",                         [0.67, 0.62, 0.65, 0.56, 0.60, 0.60]),
    ("LLaMA 3.2 (11B)",                     [0.22, 0.19, 0.21, 0.15, 0.18, 0.20]),
    ("Gemma 2 (27B)",                       [0.35, 0.51, 0.43, 0.64, 0.50, 0.56]),
    ("Pangea (7B)",                         [0.18, 0.15, 0.17, 0.10, 0.14, 0.16]),
    ("Titu-LLM",                            [0.06, 0.19, 0.08, 0.02, 0.17, 0.21]),
    ("Bong-LLaMA",                          [0.05, 0.12, 0.08, 0.02, 0.15, 0.13]),
    ("Bangla-LLaMA",                        [0.02, 0.08, 0.05, 0.10, 0.11, 0.09]),
    ("Bangla-Gemma",                        [0.18, 0.15, 0.12, 0.10, 0.22, 0.19]),
    ("TigerLLM 1B, LLaMA 3.2 (deprecated)", [0.61, 0.55, 0.68, 0.61, 0.59, 0.62]),
    ("TigerLLM-9B-it (deprecated)",         [0.72, 0.68, 0.70, 0.63, 0.65, 0.68]),
]
print(f"{len(TITULM)} comparable rows, {len(TIGER_ROWS)} context rows")

## 15. Charts

In [ ]:
if TRAIN_DONE:
    fig, ax = plt.subplots(figsize=(9.0, 3.8))
    x = np.arange(len(NAMES)); w = 0.26
    bv = [BASE_BENCH[k] for k in NAMES]
    tv = [TUNED_BENCH[k] for k in NAMES]
    qv = [TITULM["titulm-llama-3.2-1b-v2.0"].get(k) for k in NAMES]
    ax.bar(x - w, bv, w, label="gemma-3-1b-it (measured)", color=C_BASE)
    ax.bar(x,     tv, w, label=f"{MODEL_NAME} (measured)", color=C_TUNED)
    ax.bar([xi + w for xi, v in zip(x, qv) if v is not None],
           [v for v in qv if v is not None], w,
           label="titulm-1b-v2.0 (published)", color=C_OTHER)
    for xi, v in zip(x - w, bv): ax.text(xi, v+.012, f"{v:.2f}", ha="center", fontsize=7.5)
    for xi, v in zip(x,     tv): ax.text(xi, v+.012, f"{v:.2f}", ha="center", fontsize=7.5)
    for xi, v in zip(x + w, qv):
        if v is not None:
            ax.text(xi, v+.012, f"{v:.2f}", ha="center", fontsize=7.5)
    for i, n in enumerate(NAMES):
        ax.hlines(CHANCE[n], i-.45, i+.45, color="#dc2626", ls="--", lw=1.6,
                  label="random chance" if i == 0 else None)
    ax.set_xticks(x); ax.set_xticklabels(NAMES, rotation=15, ha="right")
    ax.set_ylabel("accuracy"); ax.set_ylim(0, max(bv + tv + [0.65]) * 1.3)
    ax.set_title(f"Five Bangla benchmarks, {BENCH_N:,} items each, "
                 f"log-likelihood scoring", fontsize=11, pad=10)
    ax.legend(frameon=False, fontsize=8.5, ncol=4)
    plt.savefig(f"{ASSETS}/benchmarks.png"); plt.close()

    fig, ax = plt.subplots(figsize=(5.0, 3.4))
    err = 0.5 / math.sqrt(BENCH_N * len(NAMES))
    bars = ax.bar(["base", "fine-tuned"],
                  [BASE_BENCH["average"], TUNED_BENCH["average"]],
                  color=[C_BASE, C_TUNED], width=.5,
                  yerr=[err, err], capsize=6, ecolor="#475569")
    for b, v in zip(bars, [BASE_BENCH["average"], TUNED_BENCH["average"]]):
        ax.text(b.get_x()+b.get_width()/2, v+.025, f"{v:.3f}", ha="center", fontsize=10)
    ax.set_ylabel("mean accuracy, five benchmarks")
    ax.set_ylim(0, max(BASE_BENCH["average"], TUNED_BENCH["average"]) * 1.35)
    ax.set_title("Average, with standard error", fontsize=11, pad=10)
    plt.savefig(f"{ASSETS}/average.png"); plt.close()

    fig, ax = plt.subplots(figsize=(5.6, 3.2))
    bars = ax.bar(["gemma-3-1b-it", MODEL_NAME], [BASE_PPL, TUNED_PPL],
                  color=[C_BASE, C_TUNED], width=.5)
    for b, v in zip(bars, [BASE_PPL, TUNED_PPL]):
        ax.text(b.get_x()+b.get_width()/2, v*1.01, f"{v:.1f}", ha="center", fontsize=10)
    ax.set_ylabel("perplexity (lower is better)")
    ax.set_title(f"Perplexity on {len(PPL_TEXTS)} held-out Bangla responses",
                 fontsize=11, pad=10)
    plt.savefig(f"{ASSETS}/perplexity.png"); plt.close()

    if CURVE:
        fig, ax = plt.subplots(figsize=(6.6, 3.0))
        ax.plot([s for s, _ in CURVE], [l for _, l in CURVE], color=C_TUNED, lw=1.4)
        ax.set_xlabel(f"step (of {TOTAL_STEPS:,})"); ax.set_ylabel("training loss")
        ax.set_title(f"Training loss, assistant turn only — "
                     f"{TARGET_EXAMPLES:,} examples", fontsize=11, pad=10)
        plt.savefig(f"{ASSETS}/loss.png"); plt.close()

    print("charts:", sorted(os.listdir(ASSETS)))
    from IPython.display import Image, display
    display(Image(f"{ASSETS}/benchmarks.png"))

## 16. GGUF

In [ ]:
if TRAIN_DONE:
    print(f"loading {SAVE} for export")
    model, tok = load_model(SAVE)
    if not os.path.isdir(MERGED) or not os.listdir(MERGED):
        try:
            model.save_pretrained_merged(MERGED, tok, save_method="merged_16bit")
        except Exception:
            shutil.copytree(SAVE, MERGED, dirs_exist_ok=True)
    merged_gb = sum(os.path.getsize(f"{MERGED}/{f}") for f in os.listdir(MERGED)
                    if os.path.isfile(f"{MERGED}/{f}")) / 1e9
    print(f"merged: {merged_gb:.2f} GB")

    GGUF_FILES, GGUF_ERR = [], None
    try:
        t0 = time.time()
        model.save_pretrained_gguf(GGUF, tok, quantization_method=["q4_k_m", "q8_0"])
        print(f"converted in {(time.time()-t0)/60:.0f} min")
    except Exception as e:
        GGUF_ERR = str(e)[:300]
        print("conversion raised:", GGUF_ERR)

    # Unsloth writes to "<dir>_gguf" and names files after the BASE model, so collect from
    # every plausible location and rename them after this model.
    found = sorted({p for d in [GGUF, GGUF + "_gguf", WORK]
                    if os.path.isdir(d) for p in glob.glob(os.path.join(d, "*.gguf"))})
    os.makedirs(GGUF, exist_ok=True)
    for src_f in found:
        m = re.search(r"\.(Q\d[^.]*|F16|BF16)\.gguf$", os.path.basename(src_f), re.I)
        quant = m.group(1) if m else os.path.basename(src_f).split(".")[-2]
        dst = os.path.join(GGUF, f"{MODEL_NAME}.{quant}.gguf")
        if os.path.abspath(src_f) != os.path.abspath(dst):
            shutil.move(src_f, dst)
        GGUF_FILES.append(os.path.basename(dst))
    for f in list(GGUF_FILES):          # F16 is ~2 GB and redundant next to Q8_0
        if re.search(r"\.(F16|BF16)\.gguf$", f, re.I):
            os.remove(os.path.join(GGUF, f)); GGUF_FILES.remove(f)
    GGUF_FILES = sorted(set(GGUF_FILES))
    GGUF_OK = bool(GGUF_FILES)
    for f in GGUF_FILES:
        print(f"  {f}: {os.path.getsize(os.path.join(GGUF, f))/1e9:.2f} GB")
    print("GGUF_OK =", GGUF_OK)

    json.dump({"chat_template": "gemma-3", "system": SYSTEM_BN},
              open(f"{SAVE}/prompt_format.json", "w"), indent=2, ensure_ascii=False)
    drop(model, tok)

## 17. Model card

In [ ]:
if TRAIN_DONE:
    def cl(s, n=95):
        return " ".join(str(s).split())[:n].rstrip("\\").replace("|", "\\|")

    PPL_PCT = 100 * (BASE_PPL - TUNED_PPL) / BASE_PPL

    cmp_rows = []
    for k in NAMES:
        d = TUNED_BENCH[k] - BASE_BENCH[k]
        flag = "" if abs(d) > 2 * SE else " *(noise)*"
        bl = TITULM["llama-3.2-1b (base)"].get(k)
        tl = TITULM["titulm-llama-3.2-1b-v2.0"].get(k)
        cmp_rows.append(
            f"| {k} | {CHANCE[k]:.2f} | {BASE_BENCH[k]:.3f} | **{TUNED_BENCH[k]:.3f}** | "
            f"{d:+.3f}{flag} | {bl if bl is not None else '—'} | "
            f"{tl if tl is not None else '—'} |")
    cmp_rows = "\n".join(cmp_rows)

    tiger_rows = "\n".join("| " + n + " | " + " | ".join(f"{v:.2f}" for v in vals) + " |"
                           for n, vals in TIGER_ROWS)
    ex_rows = "\n".join(f"| {cl(e['q'],58)} | {cl(e['base'])} | {cl(e['tuned'])} |"
                        for e in EXAMPLES)
    gguf_rows = ("\n".join(f"| `{f}` | {os.path.getsize(f'{GGUF}/{f}')/1e9:.2f} GB |"
                           for f in GGUF_FILES) if GGUF_OK
                 else "| — | GGUF conversion did not complete in this run |")
    split_rows = "\n".join(f"| {k} | {len(BENCH[k]):,} | {SPLITS[k]} | {CHANCE[k]:.2f} |"
                           for k in NAMES)
    census_line = (f"Of all **{DATASET_TOTAL:,}** rows, **{FULL_FIT:,}** "
                   f"({100*FULL_FIT/DATASET_TOTAL:.0f}%) fit inside the "
                   f"{MAX_SEQ}-token window."
                   if FULL_FIT else f"The dataset holds **{DATASET_TOTAL:,}** rows.")
    mode_line = " / ".join(
        f"{k}: {v.get('mode','?')}, decisions agree "
        f"{100*v.get('report',{}).get(v.get('mode','packed'),{}).get('decision_agreement',1):.0f}%"
        for k, v in SCORER_CHECK.items())

    if PPL_PCT > 10:
        headline = (f"**Perplexity on held-out Bangla falls from {BASE_PPL:.2f} to "
                    f"{TUNED_PPL:.2f}, a {PPL_PCT:.0f}% reduction.** That is the headline "
                    f"because it is the only measure here without a chance floor. The "
                    f"multiple-choice benchmarks below sit near theirs for both models and "
                    f"separate almost nothing: at {BENCH_N:,} items the standard error of a "
                    f"difference is ±{SE:.3f}, and every per-benchmark change is inside it.")
    elif PPL_PCT > 2:
        headline = (f"**Perplexity improves from {BASE_PPL:.2f} to {TUNED_PPL:.2f}**, a "
                    f"{PPL_PCT:.0f}% reduction — modest but real, and the only measure here "
                    f"with no chance floor.")
    elif PPL_PCT > -2:
        headline = (f"Perplexity is essentially unchanged ({BASE_PPL:.2f} → "
                    f"{TUNED_PPL:.2f}) and the benchmarks sit near their chance floor for "
                    f"both models. What this run establishes is the pipeline and the "
                    f"tokenizer finding below rather than a better model.")
    else:
        headline = (f"Perplexity rose from {BASE_PPL:.2f} to {TUNED_PPL:.2f}: the model "
                    f"predicts held-out Bangla worse than its base. Published as measured.")

    loss_block = ("![Training loss](assets/loss.png)\n"
                  if CURVE and os.path.exists(f"{ASSETS}/loss.png") else "")

    CARD = """---
language:
  - bn
  - en
license: gemma
library_name: transformers
pipeline_tag: text-generation
base_model: __UPSTREAM__
base_model_relation: finetune
tags:
  - bangla
  - bengali
  - gemma3
  - unsloth
  - full-finetune
  - instruction-tuned
datasets:
  - md-nishat-008/Bangla-Instruct
metrics:
  - accuracy
  - perplexity
model-index:
  - name: __MODEL_NAME__
    results:
      - task:
          type: text-generation
          name: Bangla multiple-choice understanding
        dataset:
          name: hishab Bangla benchmark suite (bangla-mmlu, commonsenseqa-bn, openbookqa-bn, piqa-bn, boolq-bn)
          type: hishab/bangla-mmlu
        metrics:
          - type: accuracy
            name: Mean accuracy across five benchmarks
            value: __AVG__
          - type: perplexity
            name: Perplexity on held-out Bangla-Instruct responses
            value: __TUNED_PPL__
---

<img src="assets/banner.png" alt="__MODEL_NAME__" width="100%">

# __MODEL_NAME__

**A full fine-tune of [`__UPSTREAM__`](https://huggingface.co/__UPSTREAM__) for Bangla.**
Every parameter was trained — not LoRA, not an adapter — on
[`Bangla-Instruct`](https://huggingface.co/datasets/md-nishat-008/Bangla-Instruct),
__SEEN__ instruction-response pairs written natively in Bangla rather than translated from
English.

__HEADLINE__

Trained on **free Kaggle T4** hardware: __STEPS__ steps over __SEEN__ examples, peaking at
__VRAM__ GB of the T4's 15 GB, across several sessions with checkpoint-resume.

## Results

Five Bangla multiple-choice benchmarks at **__BENCH_N__ items each**, scored by
**length-normalised log-likelihood** — the mean log probability of each option's tokens
given the prompt, highest wins. No generation, no LLM judge, no string matching.

![Benchmarks](assets/benchmarks.png)

| Benchmark | Chance | `gemma-3-1b-it` | **this model** | Δ | `llama-3.2-1b` † | `titulm-1b-v2.0` † |
|---|---|---|---|---|---|---|
__CMP_ROWS__
| **average** | — | __BASE_AVG__ | **__AVG__** | __D_AVG__ | — | — |

The first four columns were measured by this notebook. † marks figures **published by
Hishab** on the [`titulm-llama-3.2-1b-v2.0`](https://huggingface.co/hishab/titulm-llama-3.2-1b-v2.0)
card (0-shot), reproduced here, not re-measured.

**Same datasets, different measurement.** Those two columns are on exactly these five
datasets — TituLLM built them — which is why they are in this table at all. But they were
produced with `lm-evaluation-harness`, which feeds the raw question with no chat template
and normalises by character length, while the columns here use the model's chat template
with a Bangla system prompt and normalise by token count. Both are standard; they are not
the same measurement, and the gap between two such setups is commonly a few points. Read
the † columns for **scale**, not for ranking. A dash means Hishab did not report that cell.

What those columns do show cleanly is a pattern rather than a score: their Bangla model
lands **below** the base Llama on Bangla MMLU (0.25 vs 0.29) after 37 billion tokens of
continual pretraining. Whatever the Δ column here shows, that shape is not peculiar to
this model.

**Read the chance column before the accuracy column.** At 1B scale these benchmarks sit
close to guessing for every model in the table. The standard error of a difference between
two columns at __BENCH_N__ items is ±__SE__, so changes smaller than about __TWOSE__ carry
no information; those are marked *(noise)*.

![Average](assets/average.png)

### Benchmark provenance

| Benchmark | Items used | Splits drawn from | Chance |
|---|---|---|---|
__SPLIT_ROWS__

Three of these sets hold fewer than __BENCH_N__ rows in their test split, so splits were
concatenated to reach the target; the table says which. Three of the five
(`commonsenseqa-bn`, `openbookqa-bn`, `piqa-bn`) are **machine translations of English
datasets** refined by an LLM, per Hishab's own cards — worth knowing when reading any
Bangla score built on them.

Scoring was batched for speed and checked against an unbatched reference on real items
before use, asserting on decision agreement rather than on raw score drift
(__MODE_LINE__). Gemma-3 runs in fp32 on a T4 and its attention kernel accumulates in a
batch-shape-dependent order, so a mean log-probability moves by about 1% between batch
layouts; what must not move is the argmax, and it does not.

### Perplexity

![Perplexity](assets/perplexity.png)

| | perplexity |
|---|---|
| `gemma-3-1b-it` | __BASE_PPL__ |
| **__MODEL_NAME__** | **__TUNED_PPL__** |

Measured on __PPLN__ held-out `Bangla-Instruct` responses (shuffled rows
__PPL_FROM__–__PPL_TO__), far past the slice training drew from. Unlike the
multiple-choice tables this has no floor to sit near, which is why it leads the card.

### Appendix: figures published elsewhere, on different datasets

Read the caveats before the table, because the table is the part that invites a mistake.

**These numbers were not measured here and share no dataset with anything above.** They
come from Table 2 of the [TigerLLM paper](https://aclanthology.org/2025.acl-short.69/) as
reproduced on the [`TigerLLM-1B-it`](https://huggingface.co/md-nishat-008/TigerLLM-1B-it)
card, which evaluates on MMLU-bn (a Bangla slice of MMLU-Pro), PangBench-bn, BanglaQuaD,
mHumanEval-bn, BEnQA and BanglaRQA. Not one of those is one of the five used above. No row
for this model appears below, and none should.

**Three specific problems with it**, beyond the dataset mismatch:

1. It scores Titu-LLM at 0.06 on MMLU-bn, where Hishab's own card reports 0.25 on Bangla
   MMLU. 0.06 also sits below the 0.25 chance floor of a four-option benchmark — a figure a
   working log-likelihood harness cannot really produce.
2. It places a 1B model at 0.61 against GPT-4o-mini's 0.67 on the same benchmark.
3. The current Gemma-3 TigerLLM checkpoints carry no numbers in it at all, only a "+2–4%"
   claim relative to a deprecated LLaMA version.

The ACL version of that paper also notes its results are not reproducible.

So why reproduce it? Because "how does this compare to other Bangla models?" is a fair
question, and the honest answer is that two incompatible benchmark ecosystems exist for
Bangla and most published figures are not comparable with each other. Showing that is more
useful than silence.

| Model | __TIGER_HDR__ |
|---|__TIGER_SEP__|
__TIGER_ROWS__

None of these were re-run on the five datasets above. The 9B and 12B checkpoints do not fit
a free T4 in 16-bit, and quantising them would have produced a fourth incomparable
measurement rather than a comparison. Anyone with the hardware can settle it: the notebook
that produced this card takes a model id and the same five datasets.

### Sample generations

| Prompt | `gemma-3-1b-it` | This model |
|---|---|---|
__EX_ROWS__

Greedy decodes. They are in the card because the benchmark tables cannot answer the
question they answer. Read them before trusting any number above.

__LOSS_BLOCK__
## Tokenizer efficiency on Bangla

| | tokens per word |
|---|---|
| Bangla | __TPW__ |
| English | __ETPW__ |

Gemma-3's 262k-token vocabulary covers 140+ languages, and Bangla costs **__RATIO__×** as
many tokens per word as English here. Budget context length accordingly: a 1,024-token
window holds noticeably less Bangla than the same window of English.

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("__REPO_ID__")
model = AutoModelForCausalLM.from_pretrained("__REPO_ID__", torch_dtype=torch.float16,
                                             device_map="auto")

messages = [
    {"role": "system", "content": "__SYSTEM__"},
    {"role": "user", "content": "বাংলাদেশের মুক্তিযুদ্ধ সম্পর্কে তিনটি বাক্যে লিখুন।"},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

The Bangla system prompt was present in every training example and is in
[`prompt_format.json`](prompt_format.json). Results degrade without it.

### llama.cpp

| File | Size |
|---|---|
__GGUF_ROWS__

```bash
llama-cli -hf __GGUF_REPO__:Q4_K_M -p "রবীন্দ্রনাথ ঠাকুর কে ছিলেন?" -n 256
```

## Training

| | |
|---|---|
| Base model | `__UPSTREAM__` (__PARAMS__B parameters) |
| Method | **full fine-tune**, all __TRAINABLE__ parameters, via Unsloth |
| Dataset | `Bangla-Instruct`, __DSTOTAL__ rows available |
| Rows pulled | __SELECTN__ (shuffled, seed __SEED__) |
| Survived both filters | __POOL__ |
| **Examples trained on** | **__SEEN__**, one pass, no repeats |
| Steps | __STEPS__ at __BATCH__ × __ACCUM__ accumulation = __EFF__ effective, length-grouped |
| Max sequence length | __MAXSEQ__ tokens; longer examples dropped, not truncated |
| Learning rate | __LR__, cosine schedule, 100 warmup steps |
| Optimizer | `adamw_8bit`, `max_grad_norm=1.0` |
| Precision | __PREC__ |
| Peak VRAM | __VRAM__ GB of 15.0 |
| Hardware | __GPU__ (free Kaggle tier) |

__CENSUS_LINE__ Examples longer than that were **dropped rather than truncated**, because a
half-finished answer teaches the model to stop mid-sentence.

### Decisions that mattered

**Full fine-tuning, and why it fits.** Gemma-3-1B is roughly 2 GB in fp16 and gradients add
2 GB. Plain AdamW would need about 8 GB for its two fp32 moment buffers and would not fit a
T4; `adamw_8bit` brings the optimizer state to around 2 GB, and gradient checkpointing
absorbs the activations. At 3B this would not fit without quantising the base — at which
point it would be QLoRA, not a full fine-tune.

**Learning rate 1e-5, not 2e-4.** Every weight moves here, so steps must be small. LoRA
tolerates 2e-4 because only a fraction of a percent of parameters carry the update. A full
fine-tune at LoRA's learning rate destroys the multilingual ability the base arrived with.

**Native Bangla data, not translated Alpaca.** Most Bangla fine-tunes use machine-translated
Alpaca derivatives, which carry English sentence structure and translation artefacts into
the model. `Bangla-Instruct` was written natively: 50 volunteers at Bangladeshi
universities wrote 500 seed tasks across 10 categories, then GPT-4 and Claude 3.5 Sonnet
expanded them with self-instruct under language and cultural filters.

**Loss on the assistant turn only.** Prompt tokens are masked to `-100`, so the gradient is
not spent regenerating the question.

**Checkpoint-resume across sessions.** __SEEN__ examples do not fit in one 9-hour Kaggle
session. Each periodic checkpoint is pushed to a private repo from inside the training
loop, so the training length is set by the target rather than by the session limit, and an
interrupted session costs at most one checkpoint interval.

## Limitations

- **1B parameters.** Fluent Bangla is not accurate Bangla. This model will state false
  things confidently, particularly about dates, numbers and named people — the sample table
  above shows exactly that. Perplexity says it learned the language; those answers say it
  did not learn the facts.
- **No continual pretraining.** Instruction tuning adapts form, not knowledge. This is the
  main gap between this model and TituLLM or TigerLLM, both of which pretrained on Bangla
  text first.
- **Trained on __MAXSEQ__-token examples.** Nothing breaks past that — Gemma-3 handles 32k
  — but the fine-tune contributes nothing beyond it, and the model has a learned preference
  for answers of roughly that length. Long-form output falls back to base behaviour.
- **Multiple-choice benchmarks measure recognition, not generation.** A model can pick the
  right option and still write poor Bangla, or the reverse.
- **The benchmarks barely separate anything** at this scale, as the chance column shows.
- **No safety, bias or faithfulness evaluation** was run, in Bangla or English.
- **English ability was not measured** after fine-tuning. Hishab measured this for their
  Bangla-only model and found large English regressions; expect the same here,
  unquantified.
- **Religious, political and regional sensitivity** in Bangla content was not evaluated.
  Review output before publishing it.

## What a v2 would change

Each item addresses a specific number above, in order of expected effect:

1. **Continual pretraining on Bangla text before instruction tuning.** The largest lever by
   far, and the one every stronger Bangla model used: TituLLM 37B tokens, BongLLaMA 6.8B,
   TigerLLM 10M from school textbooks. This model used none, and no amount of extra
   instruction data substitutes for it.
2. **A larger base.** At 4B, factual capacity stops being the binding constraint, which is
   what the Bangla MMLU column is actually measuring.
3. **The remaining usable rows.** __SEEN__ of __DSTOTAL__ were used.
4. **A 1024–2048 token window**, which would also recover the rows the __MAXSEQ__-token
   filter discards. Attention cost grows with the square of length, so this trades
   throughput for coverage.
5. **Benchmarks that discriminate at this scale**, plus systematic human reading of
   generations, which is what actually matters for a writing assistant.

## Reproducing

One Kaggle notebook on a __GPU__ produced the training, every number, every chart and this
card. Raw numbers, timings and the scorer's self-check are in
[`results.json`](results.json).

## Citation

```bibtex
@inproceedings{raihan-zampieri-2025-tigerllm,
  title     = {{TigerLLM} - A Family of {B}angla Large Language Models},
  author    = {Raihan, Nishat and Zampieri, Marcos},
  booktitle = {Proceedings of ACL},
  year      = {2025}
}
@inproceedings{nahin-etal-2025-titullms,
  title     = {{TituLLMs}: A Family of {B}angla {LLM}s with Comprehensive Benchmarking},
  author    = {Nahin, Shahriar Kabir and Nandi, Rabindra Nath and Sarker, Sagor and others},
  booktitle = {Findings of ACL},
  year      = {2025}
}
@software{unsloth,
  title  = {Unsloth}, author = {Daniel Han and Michael Han and Unsloth team},
  year   = {2023}, url = {https://github.com/unslothai/unsloth}
}
```
"""

    repl = {
        "__MODEL_NAME__": MODEL_NAME, "__UPSTREAM__": UPSTREAM, "__REPO_ID__": TUNED_REPO,
        "__GGUF_REPO__": GGUF_REPO, "__SYSTEM__": SYSTEM_BN, "__HEADLINE__": headline,
        "__CMP_ROWS__": cmp_rows, "__TIGER_ROWS__": tiger_rows, "__EX_ROWS__": ex_rows,
        "__GGUF_ROWS__": gguf_rows, "__SPLIT_ROWS__": split_rows,
        "__CENSUS_LINE__": census_line, "__MODE_LINE__": mode_line,
        "__LOSS_BLOCK__": loss_block,
        "__TIGER_HDR__": " | ".join(TIGER_COLS),
        "__TIGER_SEP__": "|".join(["---"] * len(TIGER_COLS)),
        "__BENCH_N__": f"{BENCH_N:,}", "__SE__": f"{SE:.3f}", "__TWOSE__": f"{2*SE:.2f}",
        "__AVG__": f"{TUNED_BENCH['average']:.3f}",
        "__BASE_AVG__": f"{BASE_BENCH['average']:.3f}", "__D_AVG__": f"{D_AVG:+.3f}",
        "__TPW__": f"{TPW:.2f}", "__ETPW__": f"{ENTPW:.2f}",
        "__RATIO__": f"{TPW/ENTPW:.1f}",
        "__PARAMS__": f"{N_PARAMS/1e9:.1f}", "__TRAINABLE__": f"{TRAINABLE:,}",
        "__DSTOTAL__": f"{DATASET_TOTAL:,}", "__SELECTN__": f"{SELECT_N:,}",
        "__POOL__": f"{POOL:,}", "__SEEN__": f"{TARGET_EXAMPLES:,}",
        "__STEPS__": f"{TOTAL_STEPS:,}", "__MAXSEQ__": str(MAX_SEQ),
        "__LR__": f"{LR:g}", "__SEED__": str(SEED), "__BATCH__": str(BATCH),
        "__ACCUM__": str(ACCUM), "__EFF__": str(BATCH*ACCUM),
        "__PREC__": "bf16" if BF16 else "fp16",
        "__VRAM__": f"{VRAM_TRAIN:.1f}" if VRAM_TRAIN else "n/a", "__GPU__": GPU,
        "__PPLN__": str(len(PPL_TEXTS)),
        "__PPL_FROM__": f"{PPL_FROM:,}", "__PPL_TO__": f"{PPL_TO:,}",
        "__BASE_PPL__": f"{BASE_PPL:.2f}", "__TUNED_PPL__": f"{TUNED_PPL:.2f}",
    }
    for k, v in repl.items():
        CARD = CARD.replace(k, v)
    leftover = sorted(set(re.findall(r"__[A-Z0-9_]+__", CARD)))
    assert not leftover, f"unresolved placeholders: {leftover}"
    if not os.path.exists(f"{ASSETS}/banner.png"):
        CARD = re.sub(r'<img src="assets/banner\.png".*?>\n\n', "", CARD, flags=re.S)
        print("no banner.png — <img> line removed")

    open(f"{SAVE}/README.md", "w", encoding="utf-8").write(CARD)
    json.dump({
        "model": MODEL_NAME, "base_model": UPSTREAM, "bench_items": BENCH_N,
        "splits_used": SPLITS, "chance": CHANCE,
        "standard_error_of_difference": round(SE, 4),
        "scorer_self_check": SCORER_CHECK,
        "measured": {"base": BASE_BENCH, "tuned": TUNED_BENCH, "delta_average": D_AVG,
                     "perplexity": {"base": BASE_PPL, "tuned": TUNED_PPL},
                     "seconds_per_benchmark": {"base": EVAL_BASE["time"],
                                               "tuned": EVAL_TUNED["time"]}},
        "published_comparable_titulm": TITULM,
        "published_context_tigerllm": {"columns": TIGER_COLS,
                                       "rows": {n: v for n, v in TIGER_ROWS}},
        "tokens_per_word": {"bangla": round(TPW, 3), "english": round(ENTPW, 3)},
        "examples": EXAMPLES,
        "data": {"dataset_total": DATASET_TOTAL,
                 "fit_in_maxseq_whole_dataset": FULL_FIT,
                 "empty_rows": N_EMPTY, "rows_selected": SELECT_N,
                 "pool_after_filters": POOL,
                 "examples_trained_on": TARGET_EXAMPLES},
        "training": {"method": "full fine-tune via Unsloth", "max_seq": MAX_SEQ,
                     "batch": BATCH, "accum": ACCUM, "steps": TOTAL_STEPS, "lr": LR,
                     "optim": "adamw_8bit", "total_params": N_PARAMS,
                     "trainable_params": TRAINABLE,
                     "peak_vram_gb": VRAM_TRAIN, "hardware": GPU, "seed": SEED,
                     "merged_gb": round(merged_gb, 2), "gguf": GGUF_FILES,
                     "loss_curve": CURVE},
    }, open(f"{SAVE}/results.json", "w"), indent=2, ensure_ascii=False)
    print("card written\n"); print(CARD[:1400])

## 18. Push

`delete_patterns="*"` replaces the repo's file tree in a single commit, so nothing from a
previous run is left behind. The repo, its URL and its history stay.

In [ ]:
if TRAIN_DONE:
    for f in ["README.md", "results.json", "prompt_format.json"]:
        shutil.copy(f"{SAVE}/{f}", f"{MERGED}/{f}")
    shutil.copytree(ASSETS, f"{MERGED}/assets", dirs_exist_ok=True)

    create_repo(TUNED_REPO, repo_type="model", private=False, exist_ok=True)
    upload_folder(repo_id=TUNED_REPO, folder_path=MERGED, delete_patterns="*",
                  commit_message=f"{TARGET_EXAMPLES:,} examples; evaluated at "
                                 f"{BENCH_N:,} items/benchmark; mean accuracy "
                                 f"{BASE_BENCH['average']:.3f} -> "
                                 f"{TUNED_BENCH['average']:.3f}, perplexity "
                                 f"{BASE_PPL:.1f} -> {TUNED_PPL:.1f}")
    print(f"https://huggingface.co/{TUNED_REPO}")

    if GGUF_OK:
        gcard = f"""---
language: [bn, en]
license: gemma
base_model: {TUNED_REPO}
base_model_relation: quantized
pipeline_tag: text-generation
tags: [gguf, llama.cpp, bangla, bengali, gemma3]
---

# {MODEL_NAME}-GGUF

llama.cpp builds of [`{TUNED_REPO}`](https://huggingface.co/{TUNED_REPO}), a full
fine-tune of Gemma-3-1B for Bangla.

| File | Size |
|---|---|
{gguf_rows}

```bash
llama-cli -hf {GGUF_REPO}:Q4_K_M -p "রবীন্দ্রনাথ ঠাকুর কে ছিলেন?" -n 256
```

Benchmarks, baselines, provenance and limitations are on the
[model card](https://huggingface.co/{TUNED_REPO}). Quantization costs some accuracy
relative to the 16-bit weights; that cost was not separately measured.
"""
        open(f"{GGUF}/README.md", "w", encoding="utf-8").write(gcard)
        create_repo(GGUF_REPO, repo_type="model", private=False, exist_ok=True)
        upload_folder(repo_id=GGUF_REPO, folder_path=GGUF, delete_patterns="*",
                      commit_message="GGUF builds (q4_k_m, q8_0)")
        print(f"https://huggingface.co/{GGUF_REPO}")

## 19. Summary

In [ ]:
if TRAIN_DONE:
    print(f"""
Model : https://huggingface.co/{TUNED_REPO}
GGUF  : {'https://huggingface.co/' + GGUF_REPO if GGUF_OK else 'not built this run'}

dataset         {DATASET_TOTAL:,} rows -> {POOL:,} usable -> {TARGET_EXAMPLES:,} trained on
steps           {TOTAL_STEPS:,}
items/benchmark {BENCH_N:,}   (SE of a difference ±{SE:.3f})
mean accuracy   {BASE_BENCH['average']:.4f} -> {TUNED_BENCH['average']:.4f}  ({D_AVG:+.4f})
perplexity      {BASE_PPL} -> {TUNED_PPL}  ({100*(BASE_PPL-TUNED_PPL)/BASE_PPL:+.0f}%)
session         {elapsed_min():.0f} min

The private cache repo can go now: https://huggingface.co/{CACHE_REPO}/settings
Next: ONNX via onnx-community/convert-to-onnx, then a Static Space.
""")
    # from huggingface_hub import delete_repo
    # delete_repo(CACHE_REPO, repo_type="model")
else:
    print("Run again with Save & Run All to continue training.")